# 06 · Predicting Sox17 and T from BMP4 intensity, not just the mask

Every notebook so far uses only the *thresholded* `BMP4_bin` mask as the model's
input -- a 0/1 call that throws away exactly how bright a positive cell was, and
makes a cell just under threshold indistinguishable from one far under it. The
channel it was thresholded from, `BMP4_mean`, is a continuous per-cell intensity
(right-skewed, same shape as any other IF channel here) that carries strictly more
information per cell.

This notebook keeps the exact recommended recipe from 03/05 (same graph, same
architecture, same calibrated training) and changes only the **input feature set**,
comparing three:

- **mask (binary)** -- 03's baseline: `BMP4_bin` + its positive-*fraction* pyramid +
  boundary geometry (`recipe.mask_features`).
- **intensity (continuous)** -- `BMP4_mean` + a multiscale local-*mean-intensity*
  pyramid (`tools.spatial.add_intensity_pyramid`, new for this notebook),
  background-subtracted and log1p-compressed the same way `Sox17_mean`/`T_mean`
  themselves are (`recipe.intensity_features` + `build_data`'s new
  `intensity_cols=` argument) -- since this is real IF intensity, not a fraction or
  a distance, it needs the treatment `mask_features`' columns are deliberately
  exempted from. No boundary-geometry equivalent: signed distance to a mask needs a
  binary class to be inside/outside of, which a continuous channel doesn't have on
  its own.
- **mask + intensity** -- both together, in case they carry complementary
  information neither carries alone.

In [ ]:
import sys, os
sys.path.append('../src')

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools.dataset import load_well, marker_positive, IF_BACKGROUNDS, PRIMARY_WELL, MATCHED_PAIR, RESULTS
from tools.spatial import PYRAMID_SCALES
import tools.evaluation as ev
import plotting as pl

pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110

In [ ]:
import torch
from models import recipe
from models.graph import border_mask
from tools.spatial import add_mask_pyramid, add_intensity_pyramid
from tools.morphology import indices_to_mask

QUICK = False           # True -> few epochs, for a fast pass through the notebook
EPOCHS = 150 if QUICK else recipe.EPOCHS
PATIENCE = 150 if QUICK else recipe.PATIENCE
CACHE_SUFFIX = '_quick' if QUICK else ''  # keeps a provisional run from being reused as the real one

In [ ]:
df = load_well(PRIMARY_WELL)
centroids = df[['centroid_y', 'centroid_x']].to_numpy(np.float32)
IF_COLS = ['Sox17_mean', 'T_mean']

X_MASK = recipe.mask_features(df, 'BMP4_bin', max_sigma=recipe.MAX_SIGMA, geometry=True)
X_INTENSITY = recipe.intensity_features(df, 'BMP4_mean', max_sigma=recipe.MAX_SIGMA)

# (x_cols, intensity_cols) per feature set -- intensity_cols tells recipe.build_data
# which of THIS set's own columns need background-subtract + log1p (the ones built
# from BMP4_mean), as opposed to the mask-style no-op exemption every other column
# here gets.
FEATURE_SETS = {
    'mask (binary)':          (X_MASK, ()),
    'intensity (continuous)': (X_INTENSITY, tuple(X_INTENSITY)),
    'mask + intensity':       (X_MASK + X_INTENSITY, tuple(X_INTENSITY)),
}
# same slug 03/05 use for 'mask (binary)' ('mask') -- so that arm's checkpoint path
# matches theirs exactly and this notebook loads it instead of retraining
SLUGS = {'mask (binary)': 'mask', 'intensity (continuous)': 'intensity', 'mask + intensity': 'maskintensity'}
for name, (cols, _) in FEATURE_SETS.items():
    print(f'{name:24s} {len(cols):2d} features:', cols)

ceilings = pd.read_pickle(f'{RESULTS}/oracle_ceilings.pkl')
print('\noracle ceilings:', {k: {m: round(v[m],3) for m in ("r2","auroc")} for k, v in ceilings.items()})

In [ ]:
shape = (int(centroids[:,0].max())+2, int(centroids[:,1].max())+2)
valid = np.where(~border_mask(centroids, shape, recipe.RADIUS))[0]

SPLITS = {
    'random':  tuple(valid[i] for i in ev.random_split(len(valid), seed=0)),
    'blocked': tuple(valid[i] for i in ev.spatial_block_split(centroids[valid], block=800.0, seed=0)),
}
N = len(df)
MASKS = {k: tuple(indices_to_mask(i, N) for i in v) for k, v in SPLITS.items()}
for k, (tr, va, te) in SPLITS.items():
    print(f'{k:8s} train/val/test = {len(tr):,}/{len(va):,}/{len(te):,}')

## Fit all three feature sets

`mask (binary)` is cached under the same path 03/05 use (`recipe.load_or_fit`), so
it loads straight from disk if either has already run instead of retraining;
`intensity (continuous)` and `mask + intensity` are new and get cached here for the
first time.

In [ ]:
rows, fitted = [], {}
for marker in ['Sox17', 'T']:
    y_col = f'{marker}_mean'
    for split_name, (tr, va, te) in SPLITS.items():
        trm, vam, tem = MASKS[split_name]
        for fname, (cols, intensity_cols) in FEATURE_SETS.items():
            cache_path = f'{RESULTS}/models/{PRIMARY_WELL}_{SLUGS[fname]}_{marker}_{split_name}{CACHE_SUFFIX}_gnn.pt'
            model, data, log_w, history = recipe.load_or_fit(
                cache_path, df, cols, y_col, trm, vam, tr, IF_COLS, custom_background=IF_BACKGROUNDS,
                epochs=EPOCHS, patience=PATIENCE, verbose=False, intensity_cols=intensity_cols)
            pred = recipe.predict(model, data, log_w)
            y = data['y'].numpy().ravel(); ypos = data['y_positive'].numpy().ravel()
            t = tem.numpy()
            met = ev.evaluate(y[t], pred[t], ypos[t])
            rows.append(dict(target=marker, split=split_name, features=fname, **met))
            fitted[(marker, split_name, fname)] = dict(model=model, data=data, log_w=log_w, history=history,
                                                        pred=pred, y=y, ypos=ypos, metrics=met)
            print(f'{marker:6s} {split_name:8s} {fname:24s} '
                  f'R2={met["r2"]:+.4f}  AUROC={met.get("auroc", float("nan")):.3f}')

results = pd.DataFrame(rows)
results.round(4)

In [ ]:
for marker in ['Sox17', 'T']:
    sub = results[results.target == marker]
    pl.plot_metric_comparison(sub, metric='r2', group='features', hue='split',
                              ceiling=ceilings[marker]['r2'], title=f'{marker}: R2 by input feature set')
    plt.show()
    pl.plot_metric_comparison(sub, metric='auroc', group='features', hue='split',
                              ceiling=ceilings[marker]['auroc'], title=f'{marker}: AUROC by input feature set')
    plt.show()

## What the intensity pyramid looks like, next to the mask

Same smoothed-field view notebook 01 used to compare the mask against cell density
-- here comparing the binary mask against the log-intensity it was thresholded
from, at the same set of scales `recipe.intensity_features` builds its pyramid at.

In [ ]:
fig = pl.plot_field_grid(df, rows=[
    ('BMP4+ (binary mask)', 'BMP4_bin'),
    ('BMP4 intensity (log)', np.log1p(df['BMP4_mean'].to_numpy(np.float32))),
], sigmas=[30, 60, 120, 240])
plt.show()

## The two pure feature sets, side by side

Deep-diving `mask (binary)` and `intensity (continuous)` only -- `mask + intensity`
is in the comparison above if it's worth a closer look on your run, but doubling
every plot below for a third configuration is more than most readers need.

In [ ]:
for fname in ['mask (binary)', 'intensity (continuous)']:
    print(f'=== {fname} ===')
    for marker in ['Sox17', 'T']:
        out = fitted[(marker, 'blocked', fname)]
        if out['history'] is not None:
            pl.plot_training_curves(out['history'], ceiling=ceilings[marker]['r2'], title=f'{marker} ({fname})')
            plt.show()
        else:
            print(f'{marker} ({fname}): loaded from a cached checkpoint -- no training-curve history to plot')

        tem = MASKS['blocked'][2].numpy()
        fig, ax = plt.subplots(figsize=(5.8, 5.8))
        pl.plot_pred_vs_actual(out['y'][tem], out['pred'][tem], out['ypos'][tem], ax=ax,
                               title=f"{marker} ({fname}): R2={out['metrics']['r2']:.3f}  "
                                     f"AUROC={out['metrics'].get('auroc', float('nan')):.3f}")
        plt.tight_layout(); plt.show()

        pl.plot_prediction_panel(df, 'BMP4_bin', out['y'], out['pred'], target_name=f'{marker} ({fname})',
                                 mask_name='BMP4 mask (for reference)', smooth_sigma=60)
        plt.show()

## Held-out well transfer

Same check as 03: `W9_pattern1` was imaged at the same illumination power in the
same session as `W8_pattern1`, the only pair where a transfer score measures biology
rather than batch. Reusing the TRAIN-fit scaler (never refit on W9), same as 03's
own held-out-well cell.

`mask (binary)` uses the identical random-split model 03 already ran this exact
computation on -- its saved `03_transfer.csv` is reused directly below rather than
repeating the same rho computation; only `intensity (continuous)` is new here.

In [ ]:
from scipy.stats import spearmanr

df9 = load_well(MATCHED_PAIR[1])
add_mask_pyramid(df9, 'BMP4_bin', geometry=True)
add_intensity_pyramid(df9, 'BMP4_mean')

def _transfer_row(fname, marker):
    cols, intensity_cols = FEATURE_SETS[fname]
    out = fitted[(marker, 'random', fname)]
    d9 = recipe.build_data(df9, cols, f'{marker}_mean', np.arange(len(df9)),                            custom_background=IF_BACKGROUNDS, intensity_cols=intensity_cols)
    d9['x'] = torch.from_numpy(out['data']['x_scaler'].transform(df9[cols].to_numpy(np.float32)))
    p9 = recipe.predict(out['model'], d9, out['log_w'])
    rho_self = spearmanr(out['pred'], df[f'{marker}_mean']).statistic
    rho_w9 = spearmanr(p9, df9[f'{marker}_mean']).statistic
    print(f'{fname:24s} {marker:6s} rho  train={rho_self:+.3f}   held-out (W9)={rho_w9:+.3f}')
    return dict(features=fname, target=marker, rho_train_well=rho_self, rho_heldout_well=rho_w9)

try:
    mask_transfer = pd.read_csv(f'{RESULTS}/03_transfer.csv').assign(features='mask (binary)')
    print('loaded mask (binary) transfer numbers from 03_transfer.csv:')
    for _, row in mask_transfer.iterrows():
        print(f"mask (binary)            {row['target']:6s} rho  train={row['rho_train_well']:+.3f}   "
              f"held-out (W9)={row['rho_heldout_well']:+.3f}")
    transfer = mask_transfer[['features', 'target', 'rho_train_well', 'rho_heldout_well']].to_dict('records')
except FileNotFoundError:
    transfer = [_transfer_row('mask (binary)', marker) for marker in ['Sox17', 'T']]

transfer += [_transfer_row('intensity (continuous)', marker) for marker in ['Sox17', 'T']]

pd.DataFrame(transfer).round(3)

## Save

The three feature sets' checkpoints were already saved during fitting above
(`recipe.load_or_fit`) -- only the results tables need saving here.

In [ ]:
results.to_csv(f'{RESULTS}/06_markers_from_intensity.csv', index=False)
pd.DataFrame(transfer).to_csv(f'{RESULTS}/06_transfer.csv', index=False)
print(f'saved -> {RESULTS}/06_markers_from_intensity.csv')
print(f'saved -> {RESULTS}/06_transfer.csv')

## Read-out

No numbers are asserted here -- read them off the `results`/`comparison` tables and
bar charts above for your own run; this notebook is a controlled comparison, not a
verified conclusion the way 01-04 are.

A few things worth checking specifically once it's run in full (`QUICK=False`):

- Does **intensity** beat **mask** on R2/AUROC, on the **blocked** split
  specifically (the random split's spatial-autocorrelation leakage can flatter
  either feature set into looking similar even if one carries much more
  information -- see `docs/ifpredictor_methods.md` §8)?
- Does **mask + intensity** beat both individually, or does adding intensity on top
  of the mask add nothing once the mask is already there -- i.e. is the extra
  information in `BMP4_mean` mostly *redundant* with which cells cross the
  threshold, rather than adding a genuinely new signal?
- Does whichever feature set wins in-well also transfer best to `W9_pattern1`? A
  continuous channel is more exposed to batch effects (illumination power,
  exposure time, staining efficiency) than a thresholded call is BY CONSTRUCTION --
  `BMP4_bin` was built to be invariant to some of that, `BMP4_mean` was not. A
  feature set that wins in-well but transfers worse is evidence of exactly that
  trade-off, not a contradiction.
- `BMP4_mean`'s background is **auto-estimated** per column
  (`tools.qc.get_bimodal_threshold` on the train slice), unlike `Sox17_mean`/
  `T_mean`'s eye-verified `IF_BACKGROUNDS` -- if intensity underperforms
  surprisingly badly, check that estimate before concluding intensity itself is the
  problem.